# **SPECTRA training notebook**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import argparse
import yaml
import pickle
import torch
import networkx as nx
import scanpy as sc
import numpy as np
import pandas as pd
import warnings
import os

# Suppress annoying warnings for a clean console
warnings.filterwarnings("ignore")

In [3]:
# torch device setting
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(torch.version.cuda)
    print(torch.cuda.get_device_name())
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

12.1
Tesla V100-SXM2-32GB
Using device: cuda


In [4]:
from spectra.utils import set_seed
from spectra.data import data_preprocessing, build_model_dataloaders_perts_split, compute_weights, compute_weights_enhanced
from spectra.model import SPECTRA
from spectra.training import train

### **Load config file**

In [22]:
config_path = '../configs/vcc_config_final.yaml'

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)
print(yaml.dump(config, indent=2, sort_keys=False, default_flow_style=False))

project:
  name: test_reduced
  seed: 42
  deterministic: false
  use_wandb: false
data:
  adata_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/adata_vcc.h5ad
  gene_list_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/gene_list-vcc.txt
  network_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/vcc_4/VCC_4_chitin.tsv
  scgpt_embeddings_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/scGPT_embeddings_all_genes.pkl
  gene_weights_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/gene_weights-vcc.pkl
  split_path: /scratch/michele.calabro/gears/VCC/SPECTRA/data/vcc_4/VCC_4_split_indices.json
model:
  architecture: vcc_final_correct
  weights_folder_path: /group/sottoriva/michele.calabro/SPECTRA/weights/.pth
  conv_type: FAGCN
  residual_weight: 0.8
  n_channels: 24
  dropout_p: 0.05
  num_node_features: 1
training:
  batch_size: 32
  lr: 0.001
  n_epochs: 20
  alpha: 4.4
  beta: 0.005
  gamma: 2.7
  test_ratio: 0.2
  val_ratio: 0.1



In [6]:
# Initialization & Seeding - set it to deterministic for reproducibility
set_seed(seed=config['project']['seed'], deterministic=config['project']['deterministic'])

### **Load files**

In [7]:
# Load adata
adata = sc.read_h5ad(config['data']['adata_path'])
#adata.raw = adata.copy()
adata = data_preprocessing(adata,
    logtransform=True, 
    min_cells_per_pert=50)
adata

View of AnnData object with n_obs × n_vars = 221193 × 18020
    obs: 'target_gene', 'guide_id', 'batch', 'n_genes'
    var: 'gene_id', 'n_cells'
    uns: 'log1p'

In [8]:
# scGPT gene embeddings
with open(config['data']['scgpt_embeddings_path'], "rb") as f:
    scgpt_dict = pickle.load(f)

In [9]:
# load GRN
network_data = pd.read_csv(config['data']['network_path'], sep='\t')
G = nx.DiGraph()
for _, edge in network_data.iterrows():
    G.add_edge(edge['source'], edge['target'], weight=edge['weight'])

G.remove_nodes_from([n for n in G.nodes if n not in scgpt_dict])
grn_genes = set(G.nodes)

In [10]:
# Filter adata to match final network genes (gene_list)
gene_list = grn_genes & set(adata.var_names)

if grn_genes != gene_list:
    print('WARNING: some genes in the provided gene list are not included in the grn, or the gene embeddings are missing; filtering them out...')
adata = adata[:, adata.var_names.isin(gene_list)]
G = G.subgraph(gene_list).copy()

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
print("Number of nodes:", num_nodes)
print("Number of edges:", num_edges)

Number of nodes: 4922
Number of edges: 151764


In [11]:
# Filter out perturbations that aren't in the gene list
perturbations = list(adata.obs['target_gene'].unique())
perturbations.remove('non-targeting')
perts_not_included = list({
    pert for pert in perturbations
    if any(single_pert not in gene_list for single_pert in pert.split('+'))
})
if len(perts_not_included)>0:
    print('WARNING: some perturbed genes are not included in the GRN. Filtering these perturbation samples out...')
adata = adata[~adata.obs['target_gene'].isin(perts_not_included)].copy()

### **Edge indices, embeddings matrix**
Other preparation steps fro SPECTRA

In [20]:
# sanity check
assert set(G.nodes) == set(adata.var_names), "Nodes in G and adata.var_names differ!"

In [21]:
# Map Edge Index
gene_to_idx = {node: i for i, node in enumerate(adata.var_names)}
edges = [(gene_to_idx[u], gene_to_idx[v]) for u, v in G.edges()]
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

In [14]:
# Build embedding matrix
scgpt_dict = {gene_to_idx[k]: v for k, v in scgpt_dict.items() if k in gene_to_idx}
scgpt_dim = len(next(iter(scgpt_dict.values())))
embedding_matrix = torch.zeros((num_nodes, scgpt_dim))
for gene_id, emb in scgpt_dict.items():
    embedding_matrix[gene_id] = torch.tensor(emb, dtype=torch.float32)

### **Dataloaders creation**

In [24]:
perturbations = sorted(pert for pert in adata.obs["target_gene"].unique() if pert != "non-targeting")

# Merge config dictionaries for the model/dataloaders
model_config = {**config['model'], **config['training']}
model_config['dataset_size'] = adata.shape[0]
model_config['pert_to_idx'] = {pert: i for i, pert in enumerate(perturbations)}

In [25]:
from spectra.data import build_model_dataloaders_from_perts_list

train_loader, val_loader, test_loader, _, _, _, train_adata, _, test_adata = build_model_dataloaders_from_perts_list(adata, model_config, config['data']['split_path'])

Total Unique Perturbations: 148
Train set: 108 perts | 13786 cells
Val set: 20 perts | 2542 cells
Test set: 20 perts | 1908 cells


In [16]:
train_loader, val_loader, _, _, _, _, train_adata, _, _ = build_model_dataloaders_perts_split(adata, model_config)

Total Unique Perturbations: 131
Train set: 92 perts | 121883 cells
Val set: 13 perts | 11746 cells
Test set: 26 perts | 30421 cells


In [26]:
# Load WMSE Weights
weights_path = config['data']['gene_weights_path']
if os.path.exists(weights_path):
    print('found already existing gene weights dictionary! Loading...')
    with open(weights_path, 'rb') as f:
        gene_weights = pickle.load(f)
else:
    print('No gene weights dictionary found. Calculating...')
    gene_weights = compute_weights_enhanced(adata, gene_to_idx, cells_per_pert=128)
    with open(weights_path, 'wb') as f:
        pickle.dump(gene_weights, f)

found already existing gene weights dictionary! Loading...


### **Model initialization**

In [27]:
# Initialize W&B
wandb_support = config['project']['use_wandb']
if wandb_support:
    import wandb
    wandb.login()
    wandb.init(project=config['project']['name'], config=model_config)

In [28]:
model = SPECTRA(
    edge_index=edge_index, 
    num_nodes=num_nodes, 
    device=device, 
    config=model_config,
    gene_embeddings=embedding_matrix,
    gene_names=adata.var_names,
    gene_weights=gene_weights
).to(device)
print(model)

number of trainable parameters: 26713
SPECTRA(
  (gene_embeddings): Embedding(4922, 512)
  (project_gene): Linear(in_features=512, out_features=24, bias=True)
  (film_layer): GeneExpressionFiLM(
    (scale): Linear(in_features=1, out_features=24, bias=True)
    (shift): Linear(in_features=1, out_features=24, bias=True)
  )
  (add_norm): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
  (ko_mlp): MLP(
    (network): Sequential(
      (0): Dropout(p=0.0, inplace=False)
      (1): Linear(in_features=512, out_features=24, bias=True)
    )
  )
  (encoder): VariationalGraphEncoder(
    (conv1): DirFAGCNConv(24)
    (conv2): DirFAGCNConv(24)
    (ln1): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
    (ln2): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
    (proj_mu): Linear(in_features=24, out_features=24, bias=True)
    (proj_logstd): Linear(in_features=24, out_features=24, bias=True)
  )
  (gex_decoder): FeatureDecoder(
    (conv1): DirFAGCNConv(24)
    (conv2): DirFAGCN

### **Training**

In [29]:
idx_to_gene = {v: k for k, v in gene_to_idx.items()}
train(
    model=model, 
    train_loader=train_loader, 
    test_loader=val_loader,
    device=device,
    wandb_support=wandb_support,
    var_names=adata.var_names.tolist(),
    patience=7,
)

training at epoch 1: 100%|██████████| 379/379 [02:07<00:00,  2.98it/s]


training KL = 0.95615 | mse = 0.162 | cos = 0.879


testing...: 100%|██████████| 70/70 [00:06<00:00, 11.29it/s]


Validation metric improved to 0.2058. Saving best model...


training at epoch 2: 100%|██████████| 379/379 [01:56<00:00,  3.26it/s]


...

testing...: 100%|██████████| 70/70 [00:06<00:00, 11.55it/s]


No improvement. Patience: 7/7
Early stopping triggered! No improvement for 7 epochs.

--- Training concluded. Initiating Final Evaluation ---
Reloading best epoch weights for final evaluation...


100%|██████████| 16/16 [01:17<00:00,  4.84s/it]


--- Summary ---
Average Baseline AUPRC: 0.0574
Average Model AUPRC:    0.0685
FINAL TEST METRIC: 0.0685


([0.3944853263708092,
  0.1756769891700518,
  0.14367756666247009,
  0.13191580198371,
  0.1301061204170174,
  0.1283609107687165,
  0.124460800971079,
  0.12407319294903084,
  0.12466174690421464,
  0.1250892472613141,
  0.1221422748074997,
  0.12092046762047468,
  0.12199208570894905,
  0.12156381377444103,
  0.12456646828978546,
  0.12090621945647262,
  0.12345699120637295],
 [0.2024810279054301,
  0.1405792813482029,
  0.09615792866929301,
  0.06428280935755798,
  0.07559536410761732,
  0.07184640875618373,
  0.07117876524903945,
  0.06433017090229051,
  0.07095802095053451,
  0.05931651179146554,
  0.06285493786313705,
  0.05964486483218415,
  0.06074363895292793,
  0.05704514168069831,
  0.06572784690319428,
  0.0671513306110033,
  0.0679509581066668],
 [0.20581922680139542,
  0.1411315113073215,
  0.09897337495931424,
  0.07427207415457815,
  0.08553822408430278,
  0.08239497133763507,
  0.08346052700653672,
  0.07734560040989891,
  0.0825902161304839,
  0.0723929729720112,
  0.

In [ ]:
idx_to_gene = {v: k for k, v in gene_to_idx.items()}
train(
    model=model, 
    train_loader=val_loader, 
    test_loader=val_loader,
    lr=model_config['lr'], 
    n_epochs=model_config['n_epochs'],  
    device=device,
    wandb_support=wandb_support,
    var_names=adata.var_names.tolist(),
    idx_to_gene=idx_to_gene,
    alpha_weight=model_config['alpha'],
    beta_weight=model_config['beta'],
    gamma_weight=model_config['gamma'],
    eta_weight=model_config['eta']
)

In [22]:
# exit
if wandb_support:
    wandb.finish()
print('model trained and ready to go! Enjoy!')

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▂▄▅▇█
train/cosine_loss,█▄▂▂▁▁
train/global_loss,█▃▂▁▁▁
train/kl_divergence,▁▅▆▇▇█
train/mmd_ctrl,▁▁▁▁▁▁
train/mmd_pert,█▁▁▁▁▁
train/mse,█▂▂▁▁▁
val/AUPRC,▁
val/test_MMD,█▅▄▂▂▁
val/test_WMSE,▁▂▇█▂▄
epoch,6


model trained and ready to go! Enjoy!


In [ ]:
weights_dir = config['model']['weights_folder_path']
if wandb_support and wandb.run is not None:
    final_filepath = os.path.join(weights_dir,f"final_model_{model.architecture_name}_{wandb.run.id}.pth")
else:
    final_filepath = os.path.join(weights_dir, f"final_model_{model.architecture_name}_{uuid.uuid4().hex}.pth")
torch.save(model.state_dict(), final_filepath)